# Model Experiments - LSTM Stock Prediction

This notebook demonstrates model training and evaluation.

## Contents:
1. Load processed features
2. Train baseline LSTM model
3. Train combined LSTM+Sentiment model
4. Evaluate and compare models
5. Hyperparameter tuning experiments

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../src')
from models.lstm_model import LSTMPredictor
from models.combined_model import CombinedPredictor
from evaluation.evaluate import ModelEvaluator

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load Processed Data

In [ ]:
# Load feature data
df_features = pd.read_csv('../data/processed/stock_features.csv', parse_dates=['Date'])

print(f"Dataset shape: {df_features.shape}")
print(f"\nColumns: {list(df_features.columns)}")
print(f"\nTickers: {df_features['Ticker'].unique()}")

df_features.head()

In [ ]:
# Check for missing values
missing = df_features.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

## 2. Train Baseline LSTM Model

In [ ]:
# Define price features
price_features = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'MA_7', 'MA_14', 'MA_30',
    'RSI', 'MACD', 'ATR', 'Price_change'
]

# Filter to available features
available_features = [f for f in price_features if f in df_features.columns]
print(f"Using {len(available_features)} features: {available_features}")

In [ ]:
# Initialize baseline predictor
baseline_predictor = LSTMPredictor(
    sequence_length=60,
    hidden_size=64,
    num_layers=2,
    dropout=0.2,
    learning_rate=0.001,
    batch_size=32
)

# Prepare data
X, y = baseline_predictor.prepare_data(df_features, available_features, target_column='target_price')

# Split and scale
X_train, y_train, X_val, y_val, X_test, y_test = baseline_predictor.split_and_scale_data(X, y)

In [ ]:
# Build and train model
baseline_predictor.build_model(input_size=len(available_features))

train_losses, val_losses = baseline_predictor.train(
    X_train, y_train,
    X_val, y_val,
    epochs=50,  # Using fewer epochs for demonstration
    patience=10
)

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses, label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (MSE)', fontsize=12)
ax.set_title('Baseline LSTM Training History', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Evaluate Baseline Model

In [ ]:
# Make predictions
y_pred_baseline = baseline_predictor.predict(X_test)

# Evaluate
evaluator = ModelEvaluator(output_dir='../data/evaluation')
metrics_baseline = evaluator.calculate_metrics(y_test, y_pred_baseline)
evaluator.print_metrics(metrics_baseline, "Baseline LSTM")

In [ ]:
# Plot predictions
evaluator.plot_predictions(
    y_test, y_pred_baseline,
    title="Baseline LSTM Predictions",
    save_name="baseline_predictions.png"
)

In [ ]:
# Plot directional accuracy
evaluator.plot_directional_accuracy(
    y_test, y_pred_baseline,
    title="Baseline LSTM Directional Accuracy",
    save_name="baseline_directional.png"
)

## 4. Train Combined Model (if sentiment data available)

In [ ]:
# Check if sentiment data exists
import os

sentiment_file = '../data/processed/stock_features_with_sentiment.csv'

if os.path.exists(sentiment_file):
    print("Sentiment data found! Loading...")
    df_combined = pd.read_csv(sentiment_file, parse_dates=['Date'])
    
    sentiment_cols = [
        'sentiment_mean', 'sentiment_std', 'positive_mean', 
        'negative_mean', 'news_count'
    ]
    
    available_sentiment = [c for c in sentiment_cols if c in df_combined.columns]
    print(f"Available sentiment features: {available_sentiment}")
    
    has_sentiment = True
else:
    print("No sentiment data found. Skipping combined model.")
    print("Run nlp_processor.py and feature_engineer.py to generate sentiment features.")
    has_sentiment = False

In [ ]:
if has_sentiment:
    # Initialize combined predictor
    combined_predictor = CombinedPredictor(
        sequence_length=60,
        hidden_size=64,
        num_layers=2,
        dropout=0.2,
        learning_rate=0.001,
        batch_size=32
    )
    
    # Prepare data
    X_price, X_sentiment, y = combined_predictor.prepare_data(
        df_combined,
        price_columns=available_features,
        sentiment_columns=available_sentiment,
        target_column='target_price'
    )
    
    # Split and scale
    (X_price_train, X_sent_train, y_train_c,
     X_price_val, X_sent_val, y_val_c,
     X_price_test, X_sent_test, y_test_c) = combined_predictor.split_and_scale_data(
        X_price, X_sentiment, y
    )

In [ ]:
if has_sentiment:
    # Build and train
    combined_predictor.build_model(
        price_input_size=len(available_features),
        sentiment_input_size=len(available_sentiment)
    )
    
    train_losses_c, val_losses_c = combined_predictor.train(
        X_price_train, X_sent_train, y_train_c,
        X_price_val, X_sent_val, y_val_c,
        epochs=50,
        patience=10
    )

In [ ]:
if has_sentiment:
    # Plot training history
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_losses_c, label='Train Loss', linewidth=2)
    ax.plot(val_losses_c, label='Validation Loss', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss (MSE)', fontsize=12)
    ax.set_title('Combined Model Training History', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Evaluate Combined Model

In [ ]:
if has_sentiment:
    # Make predictions
    y_pred_combined = combined_predictor.predict(X_price_test, X_sent_test)
    
    # Evaluate
    metrics_combined = evaluator.calculate_metrics(y_test_c, y_pred_combined)
    evaluator.print_metrics(metrics_combined, "Combined LSTM+Sentiment")

In [ ]:
if has_sentiment:
    # Plot predictions
    evaluator.plot_predictions(
        y_test_c, y_pred_combined,
        title="Combined Model Predictions",
        save_name="combined_predictions.png"
    )

## 6. Model Comparison

In [ ]:
if has_sentiment:
    # Compare models
    results = {
        'Baseline LSTM': (y_test, y_pred_baseline),
        'Combined LSTM+Sentiment': (y_test_c, y_pred_combined)
    }
    
    comparison_df = evaluator.compare_models(results, save_name="model_comparison.png")
    
    print("\nModel Comparison Summary:")
    print(comparison_df)

## 7. Key Findings

Based on the experiments:

1. **Baseline Performance**: How well does the LSTM perform with only price data?
2. **Sentiment Impact**: Does adding sentiment improve predictions?
3. **Directional Accuracy**: Can the model predict upward/downward movements?
4. **Overfitting**: Are there signs of overfitting? Check train vs validation loss.

## Further Experiments

Ideas to improve the model:
- Try different sequence lengths (30, 90 days)
- Experiment with model architectures (more layers, different hidden sizes)
- Add attention mechanisms
- Try bidirectional LSTM
- Ensemble multiple models
- Feature engineering (add more technical indicators)

## Save Models

In [ ]:
# Save baseline model
baseline_predictor.save_model('../data/models/baseline_lstm_model.pth')
print("Baseline model saved!")

if has_sentiment:
    combined_predictor.save_model('../data/models/combined_model.pth')
    print("Combined model saved!")